In [13]:
url ='https://data.gov.il/api/3/action/datastore_search?resource_id=053cea08-09bc-40ec-8f7a-156f0677aff3'
import requests
import pandas as pd
from ipywidgets import widgets, VBox, Output
from IPython.display import display,clear_output
import matplotlib.pyplot as plt

In [14]:
pd.set_option("display.max_columns", None)
response = requests.get(url)
data = response.json()
df = pd.DataFrame(data)
data_df = pd.DataFrame(data['result']['records'])
data_df.head()
output_area = Output()

In [15]:
tozeret_nm_dropdown = widgets.Dropdown(
options=[''] + sorted(data_df['tozeret_nm'].unique().tolist()),
description='Tozeret:',
style={'description_width': 'initial'}
)

In [16]:
kinuy_mishari_dropdown = widgets.Dropdown(
options=[''],
description='Kinuy Mishari:',
style={'description_width': 'initial'}
)

In [17]:
# Function to update the second dropdown based on the selection of the first
def update_kinuy_mishari_options(change):
  if change['new']: # Check if a valid option is selected
    filtered_values = data_df[data_df['tozeret_nm'] ==
    change['new']]['kinuy_mishari'].unique()
    kinuy_mishari_dropdown.options = [''] + sorted(filtered_values)
  else:
    kinuy_mishari_dropdown.options = ['']

In [18]:
from IPython.display import clear_output

def update_output(change=None):
    with output_area:
        clear_output()

        selected_tozeret = tozeret_nm_dropdown.value
        selected_kinuy = kinuy_mishari_dropdown.value

        if selected_tozeret and selected_kinuy:
            filtered_df = data_df[
                (data_df["tozeret_nm"] == selected_tozeret) &
                (data_df["kinuy_mishari"] == selected_kinuy)
            ]

            total_records = len(filtered_df)

            unique_ramat_gimur = filtered_df["ramat_gimur"].dropna().unique()

            if len(unique_ramat_gimur) > 0:
                unique_text = ", ".join(map(str, unique_ramat_gimur))
            else:
                unique_text = "None"

            print(f"Total Records: {total_records}")
            print(f"Unique Ramat Gimur: {unique_text}")

        else:
            print("Please select valid options for both dropdowns.")

In [19]:
tozeret_nm_dropdown.observe(update_kinuy_mishari_options, names='value')
kinuy_mishari_dropdown.observe(update_output, names='value')
display(VBox([tozeret_nm_dropdown, kinuy_mishari_dropdown, output_area]))

In [20]:
df = data_df.copy()
df = df.dropna(subset=["tozeret_nm", "kinuy_mishari", "ramat_gimur", "shnat_yitzur"])

df["shnat_yitzur"] = pd.to_numeric(df["shnat_yitzur"], errors="coerce")
df = df.dropna(subset=["shnat_yitzur"])
df["shnat_yitzur"] = df["shnat_yitzur"].astype(int)

In [21]:
tab1_output = widgets.Output()
tab2_output = widgets.Output()
tab3_output = widgets.Output()

tabs = widgets.Tab(children=[tab1_output, tab2_output, tab3_output])

tabs.set_title(0, "Statistics")
tabs.set_title(1, "Data Info")
tabs.set_title(2, "Cars Per Year")

In [22]:
with tab1_output:
    clear_output()

    print("General Statistics")
    print("------------------")
    print(f"Total records: {len(df)}")
    print(f"Total columns: {len(df.columns)}")
    print()

    print("Numeric statistics:")
    display(df.describe())

In [23]:
with tab2_output:
    clear_output()

    info_df = pd.DataFrame({
        "Column Name": df.columns,
        "Data Type": df.dtypes.astype(str),
        "Missing Values": df.isnull().sum().values,
        "Unique Values": df.nunique().values
    })

    display(info_df)

In [24]:
with tab3_output:
    clear_output()

    cars_per_year = df["shnat_yitzur"].value_counts().sort_index()

    plt.figure(figsize=(12, 5))
    plt.plot(cars_per_year.index, cars_per_year.values, marker="o")
    plt.title("Number of Cars Created Each Year")
    plt.xlabel("Year")
    plt.ylabel("Number of Cars")
    plt.grid(True)
    plt.show()

In [25]:
display(tabs)